In [4]:
import pandas as pd
import numpy as np
from google import genai
from google.genai import types
import json
import base64
from typing import Dict
import os
from anthropic import Anthropic

google_api_key = os.getenv("GOOGLE_API_KEY")
client_google = genai.Client(api_key = google_api_key)
model_google = "gemini-2.5-flash"

claude_api_key = os.getenv("ANTHROPIC_API_KEY")
client_claude = Anthropic(api_key = claude_api_key)
model_claude = "claude-sonnet-4-5-20250929"

In [7]:
def create_image(filepath):
    with open(filepath, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

In [8]:
test_cases = {
    "julpa": {
        "images": [
            "quine_images/bank_vault_door.jpg",
            "quine_images/body_of_a_door.jpg", 
            "quine_images/doorknob.jpg",
            "quine_images/glass_door.jpg",
            "quine_images/white_door.jpg"
        ],
        "options": {
            "a": "door",
            "b": "entrance",
            "c": "barrier",
            "d": "opening",
            "e": "portal"
        }
    },
    "blirox": {
        "images": [
            "quine_images/deer.jpg",
            "quine_images/deers_fur.jpg",
            "quine_images/deer_running.jpg",
            "quine_images/deers_hooves.jpg",
            "quine_images/deers_head.jpg"
        ],
        "options": {
            "a": "deer",
            "b": "temporal slices of deeer",
            "c": "brown",
            "d": "animal",
            "e": "fur"
        }
    }
}

def run_experiment(word, test_case, n_iterations=10):
    """Run experiment for one word"""
    
    options_text = "\n".join([f"{k}) {v}" for k, v in test_case["options"].items()])
    
    prompt = f"""Imagine you are a linguist that has to create a translation manual for a language of a remote tribe. 
                You have to find a proper translation for the object that the native points to with his finger and exclaims {word} upon seeing it.
                You cannot assume any prior framework for individuation of sense experience and any linguistic presuppositions.
                    
                Is the proper translation: {options_text}

                Answer with just the letter.
                """

    results = []
    
    for i in range(n_iterations):
        contents = [prompt]
        for img_path in test_case["images"]:
            img_data = create_image(img_path)
            contents.append({
                "inline_data": {"mime_type": "image/jpeg", "data": img_data}
            })
        
        response = client_google.models.generate_content(
            model=model_google,
            contents=contents,
            config={"temperature": 0.7}
        )
        
        results.append(response.text)
        print(f"{word} - Trial {i+1}: {response.text}")
    
    return results

In [9]:
all_results = {}
for word, test_case in test_cases.items():
    print(f"\n{'='*80}\nTesting word: {word}\n{'='*80}")
    all_results[word] = run_experiment(word, test_case, n_iterations = 40)


Testing word: julpa
julpa - Trial 1: a) door
julpa - Trial 2: A
julpa - Trial 3: A
julpa - Trial 4: c)
julpa - Trial 5: A
julpa - Trial 6: c)
julpa - Trial 7: a)
julpa - Trial 8: The correct answer is **c) barrier**.

Here's why:

The instruction "You cannot assume any prior framework for individuation of sense experience and any linguistic presuppositions" is crucial. This means you, as the linguist, cannot assume that the native speaker categorizes the world into discrete objects like "doors" in the same way an English speaker does. You are observing a single instance of a word being used.

Let's evaluate the options based on this constraint:

*   **a) door**: While the object in the image (a vault door) is indeed a door, translating "julpa" as "door" makes a very specific claim about the native's categorization. It assumes that the native's concept of "julpa" perfectly aligns with the English concept of "door" – a movable panel that opens and closes an entrance. This is a strong as

In [39]:
def LLM_response(filepath, prompt):

    def create_image(filepath):
        with open(filepath, "rb") as image_file:
            return base64.b64encode(image_file.read()).decode('utf-8')
        
    image = create_image(filepath)


    response = client_google.models.generate_content(
                    model=model_google,
                    contents=[
                        {"role": "user", 
                         "parts": [
                             {"text": prompt},
                {"inline_data": {"mime_type": "image/jpeg", "data": image}} 
        ]
                        }
                    ]
    )
    print(f"Result: {response.text}\n")

In [51]:
quine_prompt = """Imagine you are a linguist that has to create a translation manual for a language of a remote tribe. 
                You have to find a proper translation for the object that the native points to with his finger.
                    
                Is the proper translation: a) door, b) entrance, c) rectangular barrier, d) portal, e) room?

                Answer with just the letter and brief explanation.
                """

LLM_response("quine_images/bank_vault_door.jpg", quine_prompt)

Result: A) door.
This object functions as a movable barrier designed to allow or block passage, which is the universal definition of a door, regardless of its specific construction or purpose (like a vault door). The other options are either too vague (rectangular barrier), describe the opening rather than the object (entrance, portal), or are completely incorrect (room).



In [52]:
quine_prompt = """Imagine you are a linguist that has to create a translation manual for a language of a remote tribe. 
                You have to find a proper translation for the object that the native points to with his finger.
                    
                Is the proper translation: a) door, b) entrance, c) rectangular barrier, d) portal, e) room?

                Answer with just the letter and brief explanation.
                """

LLM_response("quine_images/body_of_a_door.jpg", quine_prompt)

Result: a) door.

**Explanation:** As a linguist, the goal is to find the most precise and functionally equivalent term. The object shown is a panel, typically made of wood or similar material, that opens and closes to allow or block passage. This is universally known as a "door."
*   "Entrance" refers to the opening itself, not the movable barrier.
*   "Rectangular barrier" is a descriptive attribute but not the specific name of the object.
*   "Portal" can be synonymous with door, but often implies a grander or more symbolic entrance; "door" is more common and precise for this everyday object.
*   "Room" refers to the space beyond the object, not the object itself.



In [53]:
quine_prompt = """Imagine you are a linguist that has to create a translation manual for a language of a remote tribe. 
                You have to find a proper translation for the object that the native points to with his finger.
                    
                Is the proper translation: a) door, b) entrance, c) rectangular barrier, d) portal, e) room?

                Answer with just the letter and brief explanation.
                """

LLM_response("quine_images/white_door.jpg", quine_prompt)

Result: a) door.

The object pointed to is the movable panel that separates rooms and swings on hinges, which is universally recognized as a door. The other options are either too general (rectangular barrier), refer to the opening rather than the object (entrance), carry different connotations (portal), or refer to the space beyond (room). As a linguist, one aims for the most direct and accurate functional equivalent.

